In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("AZURE_OPENAI_API_KEY")
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")

client = OpenAI(
    base_url=endpoint,
    api_key=api_key
)

# 2.1 - Experimentación con temperature

Prueba el mismo prompt con diferentes valores de temperature:

- temperature = 0.0
- temperature = 0.5
- temperature = 1.0
- temperature = 1.5 (si la API lo permite)

Analiza cómo cambian las respuestas. Usa un caso práctico (ejemplo: generar un eslogan, escribir código, responder una pregunta técnica).

In [6]:
prompt_usuario = "Dame solo un nombre creativo para una cafetería que solo abre de noche."

temperaturas = [0.0, 0.5, 1.0, 1.5]

for temp in temperaturas:
    print(f"\n--- Probando con Temperature: {temp} ---")
    
    response = client.chat.completions.create(
        model=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
        messages=[
            {"role": "system", "content": "Eres un experto en branding y marketing creativo."},
            {"role": "user", "content": prompt_usuario}
        ],
        temperature=temp,
        max_tokens=50
    )
    
    print(f"Respuesta: {response.choices[0].message.content.strip()}")


--- Probando con Temperature: 0.0 ---
Respuesta: "Noche de Café"

--- Probando con Temperature: 0.5 ---
Respuesta: "Estrella Café Nocturno"

--- Probando con Temperature: 1.0 ---
Respuesta: "Nochecafé"

--- Probando con Temperature: 1.5 ---
Respuesta: "Estrella Nocturna"


**Análisis:**
- Temp 0.0: Nombre muy simple, dice literalmente lo que es.
- Temp 0.5: Mete estrella porque se ven por la noche.
- Temp 1.0: Comprime el primer nombre, sin mucho sentido porque se hace difícil de leer y no va a calar.
- Temp 1.5: No tiene mucho sentido, podría ser cualquier tipo de negocio (o ni si quiera un negocio) que abra por la noche.

# 2.2 - Experimentación con top_p

Prueba el mismo prompt con diferentes valores de top_p:

- top_p = 0.1
- top_p = 0.5
- top_p = 0.9
- top_p = 1.0

Mantén temperature = 1.0 para ver el efecto puro de top_p. Compara con los resultados de temperature.

In [9]:
prompt_usuario = "Dame dos nombres creativos para una cafetería que solo abre de noche."

top_p = [0.1, 0.5, 0.9, 1.0]

for t_p in top_p:
    print(f"\n--- Probando con Top P: {t_p} ---")
    
    response = client.chat.completions.create(
        model=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
        messages=[
            {"role": "system", "content": "Eres un experto en branding y marketing creativo."},
            {"role": "user", "content": prompt_usuario}
        ],
        temperature=1.0,
        top_p=t_p,        
    )
    
    print(f"Respuesta: {response.choices[0].message.content.strip()}")


--- Probando con Top P: 0.1 ---
Respuesta: ¡Claro! Aquí tienes dos nombres creativos para una cafetería nocturna:

1. **Café Nocturno**: Este nombre evoca la idea de un lugar acogedor y misterioso donde disfrutar de una buena taza de café bajo el manto de la noche.

2. **Luz de Luna Café**: Este nombre sugiere un ambiente mágico y relajante, ideal para aquellos que buscan un refugio nocturno para disfrutar de su bebida favorita.

Ambos nombres transmiten la esencia de un espacio único y especial que solo se vive en la noche.

--- Probando con Top P: 0.5 ---
Respuesta: ¡Claro! Aquí tienes dos nombres creativos para una cafetería nocturna:

1. **Café Nocturno**: Un nombre que evoca la atmósfera mágica de la noche, ideal para un lugar donde los clientes pueden disfrutar de su café bajo las estrellas.

2. **Luz de Luna Café**: Este nombre sugiere un ambiente acogedor y misterioso, perfecto para aquellos que buscan un refugio nocturno para relajarse y disfrutar de una buena taza de café.



**Análisis**

1. De 0.1 a 0.9 las respuestas son casi idénticas. Esto ocurre porque para el modelo, los nombres "Café Nocturno" y "Luz de Luna Café" tienen una probabilidad tan alta que, incluso cuando abres el filtro al 0.9, esos dos nombres siguen estando arriba del todo. Al usar top_p, el modelo suma las probabilidades de las palabras más lógicas. Si "Café" (0.6) + "Nocturno" (0.3) ya suman 0.9, el modelo borra cualquier otra opción más creativa porque no entra en ese top_p.

2. Cuando se aumenta a 1.0, no se filtra nada, por lo que el modelo se permitió elegir "Noche de Café". Lo que indica que "Noche de Café" estaba abajo en la lista. Al filtrar aunque fuera un poquito, esa opción desaparecía.

# 2.3 - Experimentación con Penalties

Prueba prompts que tiendan a repetir contenido (por ejemplo: describir un producto, generar múltiples ideas similares) con:

- presence_penalty = 0.0 vs presence_penalty = 0.6
- frequency_penalty = 0.0 vs frequency_penalty = 0.8
- Combinación de ambos penalties

Analiza qué tipo de repeticiones evita cada uno.

In [13]:
import time

prompt_usuario = "Describe una manzana roja en tres párrafos, centrándote mucho en su color, su sabor dulce y su textura crujiente."

pres_penalties = [0.0, 0.6]
freq_penalties = [0.0, 0.8]

for pp in pres_penalties:
    for fp in freq_penalties:
        print("\n" + "="*60)
        print(f"EJECUTANDO: Presence Penalty: {pp} | Frequency Penalty: {fp}")
        print("="*60)
        
        try:
            response = client.chat.completions.create(
                model=deployment_name,
                messages=[
                    {"role": "system", "content": "Eres un experto en branding y marketing creativo."},
                    {"role": "user", "content": prompt_usuario}
                ],
                presence_penalty=pp,
                frequency_penalty=fp,
                temperature=0.7
            )
            
            resultado = response.choices[0].message.content.strip()
            print(f"Respuesta recibida:\n\n{resultado}\n")
            
            # Un pequeño respiro para evitar saturar la conexión
            time.sleep(1) 
            
        except Exception as e:
            print(f"Error en esta combinación: {e}")

print("\n--- Experimento finalizado con éxito ---")


EJECUTANDO: Presence Penalty: 0.0 | Frequency Penalty: 0.0
Respuesta recibida:

La manzana roja es un deleite visual que evoca sensaciones de frescura y vitalidad. Su piel, de un vibrante color rojo, brilla bajo la luz, capturando la atención de cualquiera que se acerque. Este tono rojo profundo puede variar desde un carmesí intenso hasta matices más suaves, pero en cada caso, la manzana irradia una belleza que parece prometer una experiencia gustativa excepcional. Este color no solo es atractivo, sino que también simboliza la frescura y la naturalidad, sugiriendo que lo que hay dentro es igual de irresistible.

Al morder una manzana roja, el primer impacto es la explosión de su dulce sabor que inunda el paladar. La dulzura es equilibrada, sin ser empalagosa, lo que la convierte en un bocado perfecto para saciar cualquier antojo. Este sabor, a menudo descrito como una mezcla de miel y caramelo, se complementa con notas sutiles que pueden recordar a la canela o a frutos del bosque, ofr

1. TEST PP 0.0 | FP 0.0
- Análisis: Es la respuesta estándar del modelo. Correcta, pero predecible.
- Repeticiones: Usa "manzana roja" al principio de cada sección prácticamente. Repite palabras como "vitalidad" y "frescura" en varios párrafos.
- Vocabulario: Usa los adjetivos más probables (rojo, dulce, crujiente).
- Sensación: Suena a descripción de enciclopedia o de catálogo genérico. Es un texto "seguro" pero sin alma.

2. TEST: PP 0.0 | FP 0.8
- Análisis: Aquí el frequency_penalty ha forzado al modelo a buscar palabras menos comunes para no ser multado.
- Vocabulario: En lugar de solo "rojo", usa "carmesí y rubí". En lugar de "piel", usa "superficie lustrosa". Cambia "dulce" por "abrazo del verano encapsulado".
- Efecto: El texto se vuelve mucho más rico visualmente. Se nota que el modelo se "esfuerza" en su léxico para evitar los tokens que ya usó en las primeras líneas.
- Resultado: Ideal si quieres que la IA escriba con un vocabulario más culto o variado.

3. TEST: PP 0.6 | FP 0.0
- Análisis: El presence_penalty castiga al modelo si intenta volver a temas o palabras que ya aparecieron.
- Estructura: El modelo intenta que los párrafos no se parezcan en nada. Se centra más en la experiencia que en la descripción estática.
- Conceptos: Introduce ideas nuevas como "símbolo de vitalidad" o "viaje de placer".
- Fallo: Al tener el frequency_penalty en 0.0, todavía se permite repetir palabras comunes como "manzana roja" o "dulce" varias veces, porque aunque el tema cambie, las palabras individuales no están penalizadas por frecuencia.

4. TEST: PP 0.6 | FP 0.8
- Análisis: Es el texto más sofisticado de los cuatro. Es la combinación de variedad léxica y progresión temática.
- Lo mejor: El último parrafo en lugar de solo decir que es crujiente, dice: "Esta firmeza inicial se transforma rápidamente en suavidad al degustarla, creando un contraste perfecto".
- Evolución: Ha pasado de ser una lista de adjetivos en el primer caso a ser una narrativa sensorial.
- Evitación de muletillas: Casi no hay repetición de palabras "vacías". Cada frase aporta algo nuevo y usa términos como "momento lúdico" o "rubí luminoso" que no aparecían en el primer test.

# 2.4 - Preguntas Teóricas (Responder con tus propias palabras)

## 1. ¿Cuál es la diferencia entre top_p y temperature?

La diferencia es que temperature lo hace más o menos determinista y top_p no lo cambia, simplemente alarga o acorta la lista de palabras probables.

## 2. ¿Por qué no se recomienda ajustar top_p y temperature al mismo tiempo?

- Contrariedad: Si son contrarios (temp: 1.0 / top_p: 0.1) el efecto de ambos parámetros podría anularse uno respecto del otro.
- Redundancia: Con ajustar uno de los dos sería suficiente para conseguir la aleatoriedad desdeada. Es mejor dejer uno por defecto y tocar el otro.

## 3. ¿Cuál es la diferencia entre presence_penalty y frequency_penalty?

- Presence Penalty: Penaliza a un token si ya ha aparecido, haciendo al modelo cambiar de tema.
- Frequence Penalty: Penaliza a un token basándose en cuantas veces ha aparecido, haciendo al modelo buscar sinónimos para no repetir palabras.